# Producción Agrícola (Procesamiento)

En esta libreta prepararemos los datos para el futuro analisis exploratorio de datos.

In [1]:
from scripts.config import PRELOADS
import pickle

# Notemos que ahora estamos obteniendo la información ya normalizada.
with open(PRELOADS['AGROPROD_1'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

dgsiap

En esta etapa (/interm a /process) nos permitimos modificar el diccionario y los datos para mejorar su uso para el analisis de datos.

## Datos

Comenzaremos con los conjuntos de datos. En particular la corrección que nos quedo pendiente en la etapa anterior. Esto porque antes no nos permitiamos modificar los datos. Ahora si.

**Observación** No confundir lo ocurrido con la columna Precio que si fue modificada. La metodologia en la columna Precio no cambio. Con la columna pendiente si. 

In [60]:
# Revisamos los conjuntos de datos que tengan y no tengan la columna pendiente.
for file in dgsiap.branch:
    df = file.open(pars=[1, {"low_memory":False, "index_col":0}])
    print(df.columns.tolist()[14:18])

# Elegimos el conjunto de referencia (el que ya quito las unidades) y el problemático a normalizar.
check = dgsiap.branch[0]
df_check = check.open(pars=[1, {"low_memory":False, "index_col":0}])
base = dgsiap.branch[6]
df_base= base.open(pars=[1, {"low_memory":False, "index_col":0}])

['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo Sin Um', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['Nomunidad', 'Idcultivo', 'Nomcultivo', 'Sembrada']
['No

Seleccionando uno de cada uno. Ahora debemos crear una asignación con respecto al conjunto base para transformar la problemática.

In [70]:
crops = df_base['Nomcultivo Sin Um'].unique().tolist() # Seleccionamos los que ya esten bien escritos.
df_prob = df_check[~df_check['Nomcultivo'].isin(crops)] # Quitamos los que coinciden con la base de referencia.
crops_prob=df_prob['Nomcultivo'].unique().tolist()
crops_prob[:10]

['Avena forrajera achicalada',
 'Cebada forrajera achicalada',
 'Pasto tapete (m2)',
 'Girasol flor (gruesa)',
 'Flores',
 'Palma de ornato (planta)',
 'Clavel (gruesa)',
 'Eucalipto',
 'Flor cera',
 'Pastos y praderas achicalado']

Vemos entonces cosas raras como m2, gruesa y/o planta. Entonces, debemos corroborar si se trata de una aclaración o las unidades del cultivo.

In [93]:
test = df_prob[['Nomunidad','Nomcultivo']]

import re

test['uni_s'] = test['Nomunidad'].str.lower()
test['cul_s'] = test['Nomcultivo'].str.lower()

test['Par_Regex'] = [re.search(r"\(.*\)", v).group(0) if re.search(r"\(.*\)", v) else "" for v in test['cul_s'] ]
sub_afre = test[test['Par_Regex']!=""]
sub_afre['unidad_par'] = [v if v==p[1:-1] else "" for v,p in zip(sub_afre['uni_s'], sub_afre['Par_Regex'])]
goal = sub_afre[sub_afre['unidad_par']!=""]
goal2 = sub_afre[sub_afre['unidad_par']==""]
print(goal2.head(n=5))
print(goal)

           Nomunidad         Nomcultivo           uni_s              cul_s  \
315   Metro cuadrado  Pasto tapete (m2)  metro cuadrado  pasto tapete (m2)   
799   Metro cuadrado  Pasto tapete (m2)  metro cuadrado  pasto tapete (m2)   
6893  Metro cuadrado  Pasto tapete (m2)  metro cuadrado  pasto tapete (m2)   
9552  Metro cuadrado  Pasto tapete (m2)  metro cuadrado  pasto tapete (m2)   
9603  Metro cuadrado  Pasto tapete (m2)  metro cuadrado  pasto tapete (m2)   

     Par_Regex unidad_par  
315       (m2)             
799       (m2)             
6893      (m2)             
9552      (m2)             
9603      (m2)             
      Nomunidad                Nomcultivo   uni_s                     cul_s  \
335      Gruesa     Girasol flor (gruesa)  gruesa     girasol flor (gruesa)   
367      Planta  Palma de ornato (planta)  planta  palma de ornato (planta)   
380      Gruesa     Girasol flor (gruesa)  gruesa     girasol flor (gruesa)   
409      Gruesa           Clavel (gruesa)  grue

Con lo anterior vemos que deberemos estar al pendiente con las abreviaturas. Así que eso lo haremos de forma circunstancial. Ahora nos enfocaremos en las que indican de forma explicita la unidad que estan usando.

In [115]:
def norm_um(df):
    crops = df['Nomcultivo'].unique().tolist()
    um = df['Nomunidad'].unique().tolist()
    um = set([u.lower() for u in um])

    par_detect = {v: re.search(r"\(.*\)", v.lower()).group(0) for v in crops if re.search(r"\(.*\)", v)}
    # print(par_detect)
    crop_w_um = []
    for crop in par_detect.keys():
        if par_detect[crop][1:-1] in um:
            crop_w_um.append(crop)
    df['Nomcultivo'] = df['Nomcultivo'].apply(lambda x: re.sub(r"\(.*\)","", x) if x in set(crop_w_um) else x)
    return df
    

fase1_t = norm_um(df_check)
fase1_t

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.00,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.00,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.00,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.20,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.00,16.33,5285.21,2589752.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35897,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,7960000,Nuez,10.0,10.0,0.0,33.00,3.30,55000.00,1815000.00
35898,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,9130000,Uva,1.0,1.0,0.0,9.90,9.90,11444.38,113299.36
35899,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,15050000,Pastos y praderas,36.0,36.0,0.0,1396.96,38.80,963.14,1345470.00
35900,2025,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,5050000,Agave,260.0,170.0,0.0,14960.00,88.00,1200.00,17952000.00


Ya terminamos con la parte principal. Verificamos y revisamos la parte secundaria.

In [117]:
test = fase1_t[['Nomunidad','Nomcultivo']]
import re

test['uni_s'] = test['Nomunidad'].str.lower()
test['cul_s'] = test['Nomcultivo'].str.lower()

test['Par_Regex'] = [re.search(r"\(.*\)", v).group(0) if re.search(r"\(.*\)", v) else "" for v in test['cul_s'] ]
sub_afre = test[test['Par_Regex']!=""]
sub_afre['unidad_par'] = [v if v==p[1:-1] else "" for v,p in zip(sub_afre['uni_s'], sub_afre['Par_Regex'])]
goal = sub_afre[sub_afre['unidad_par']!=""]
goal2 = sub_afre[sub_afre['unidad_par']==""]
print(goal)  # Vemos que las columnas principales ya fueron adaptadas.
goal2

Empty DataFrame
Columns: [Nomunidad, Nomcultivo, uni_s, cul_s, Par_Regex, unidad_par]
Index: []


,Nomunidad,Nomcultivo,uni_s,cul_s,Par_Regex,unidad_par
11,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
44,Tonelada,Col (repollo),tonelada,col (repollo),(repollo),
52,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
74,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
102,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
...,...,...,...,...,...,...
35820,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
35842,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
35849,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),
35874,Tonelada,Tomate rojo (jitomate),tonelada,tomate rojo (jitomate),(jitomate),


Vemos que solo quedan aclaraciones. Además de las unidades que hayan abreviado. Para revisar eso haremos el ajuste en todo el conjunto de datos.

In [128]:
def checkp2(df):
    # Revisamos las unidades que no tengan el problema principal.
    test = df[['Nomunidad','Nomcultivo']]
    test['uni_s'] = test['Nomunidad'].str.lower()
    test['cul_s'] = test['Nomcultivo'].str.lower()

    test['Par_Regex'] = [re.search(r"\(.*\)", v).group(0) if re.search(r"\(.*\)", v) else "" for v in test['cul_s'] ]
    sub_afre = test[test['Par_Regex']!=""]
    sub_afre['unidad_par'] = [v if v==p[1:-1] else "" for v,p in zip(sub_afre['uni_s'], sub_afre['Par_Regex'])]
    goal = sub_afre[sub_afre['unidad_par']!=""]
    goal2 = sub_afre[sub_afre['unidad_par']==""]
    # print(goal)  # Vemos que las columnas principales ya fueron adaptadas.
    print(goal2['Nomunidad'].unique().tolist())

ndf = []

for file in dgsiap.branch:
    df = file.open(pars=[1, {"low_memory":False, "index_col":0}])
    cols = set(df.columns)
    if "Nomcultivo" in cols:
        vdf = norm_um(df)
        ndf.append(vdf)
        checkp2(vdf)
    else:
        ndf.append(df)


['Tonelada', 'Metro cuadrado']
['Tonelada', 'Metro cuadrado']
['Tonelada', 'Metro cuadrado']
['Tonelada', 'Metro cuadrado']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada', 'Metro cuadrado']
['Tonelada', 'Metro cuadrado']


Vemos que el metro cuadrado es la única unidad problematica que falta resolver. Esto porque en la base de datos la abrevian. Entonces haremos el ajuste manual dentro de norm_um

In [291]:
ndf = []
for file in dgsiap.branch:
    df = file.open(pars=[1, {"low_memory":False, "index_col":0}])
    cols = set(df.columns)
    if "Nomcultivo" in cols:
        vdf = norm2_um(df)
        ndf.append(vdf)
        checkp2(vdf)
    else:
        ndf.append(df)


['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']
['Tonelada']


Viendo finalmente que la unica restante es la tonelada. La unidad principal del conjunto de datos. Entonces ya podemos hacer la normalización final con respecto al diccionario de datos.

In [292]:
def posnorm_branch(tree, norm_col="Nomcultivo", norm_rule=norm2_um):
    codes = tree.root
    catalog = codes.open(pars=[1, {"index_col":0}])
    ndf = []
    for file in tree.branch:
        df = file.open(pars=[1, {"low_memory":False, "index_col":0}])
        cols = set(df.columns)
        if norm_col in cols:
            vdf = norm_rule(df)
            ndf.append(vdf)
        else:
            df.columns = catalog.index
            ndf.append(df)
    return ndf

dgsiap_normb = posnorm_branch(dgsiap)  # Obteniendo el conjunto de datos a utilizar

## Diccionario de Datos.

Una vez resuelto el problema de las columnas continuamos con el Diccionario de Datos. Modificando las reglas definidas por la fuente oficial. Para ello ocuparemos el catalogo y los tipos de datos definidas.

In [293]:
codes = dgsiap.root
data_vars = dgsiap.subroot

catalog = codes.open(pars=[1, {"index_col":0}])
print(data_vars)
catalog

{'T': 'Texto', 'N': 'Numérico entero', 'D': 'Numérico decimal', '#': 'Carácter numérico.', 'A': 'Carácter alfanumérico.'}


,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
Nombre de campo,,,,,
Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
Idestado,N,2,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
Nomestado,T,60,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
Idddr,N,3,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
Nomddr,T,60,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
Idcader,N,2,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
Nomcader,T,90,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
Idmunicipio,T,3,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
Nommunicipio,T,45,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...


### Correccion de Tipos
Para nuestro diccionario usaremos en gran parte los tipos de datos dados por la libreria pandas. Para ello usaremos el primer conjunto de datos como referencia y forzaremos el tipo en el resto.


In [152]:
test = dgsiap.branch[0]
df = test.open(pars=[1, {"index_col":0, "low_memory":False}])
df.dtypes

Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

Primero definiremos el tipo de dato de pandas que usará cada variable a partir de ahora.

In [153]:
import pandas as pd

agro_varsguide = pd.DataFrame(index = data_vars.keys(), columns = ['Tipo Pandas', 'Descripción'])

for col in df.columns:
    # print(col)
    var_type = catalog.loc[col, 'Tipo']
    agro_varsguide.loc[var_type] = [df.dtypes[col], data_vars[var_type]]

agro_varsguide=agro_varsguide.dropna(axis=0)
agro_varsguide

,Tipo Pandas,Descripción
T,str,Texto
N,int64,Numérico entero
D,float64,Numérico decimal


Ahora haremos la conexión con las columnas.

In [154]:
# Comenzamos con la conexión general.
agro_dtypes = []
for col in df.columns:
    var_type = catalog.loc[col, 'Tipo']
    agro_dtypes.append(agro_varsguide.loc[var_type, 'Tipo Pandas'])


data_checker = pd.Series(index = df.columns, data=agro_dtypes)
data_checker

Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio               str
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

# Casos especiales.

Como se comentó en la etapa anterior. Idmunicipio no tiene argumentos para ser de tipo texto cuando el resto de identificadores son tipo entero. Entonces haremos el cambio.

In [155]:
data_checker['Idmunicipio'] = agro_varsguide.loc['N', 'Tipo Pandas']
data_checker

Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

Ya tenemos los tipos definidos. Ahora falta corroborar que estos coinciden con el resto.

In [204]:
# print(data_checker)
toparse_dfs = []
for k, df in enumerate(dgsiap_normb):
    dft = df.dtypes
    diff = dft[data_checker!=dft]
    if len(diff)>0:
        print("Conjunto de datos:",k)
        print("Columnas problematicas:", diff.index.tolist())

Conjunto de datos: 3
Columnas problematicas: ['Volumenproduccion']
Conjunto de datos: 4
Columnas problematicas: ['Sembrada']
Conjunto de datos: 16
Columnas problematicas: ['Rendimiento']


Vemos entonces que tenemos 3 columnas en df distintos tal que los tipos de dato no coinciden con los tipos de datos de referencia.

In [208]:
df1 = dgsiap_normb[3]
print(df1['Volumenproduccion'].dtype)
df2 = dgsiap_normb[4]
print(df2['Sembrada'].dtype)
df3 = dgsiap_normb[16]
print(df3['Rendimiento'].dtype)

str
str
str


Una vez confirmadas las columnas problemáticas procedemos con el ajuste.

In [265]:
df1 = dgsiap_normb[3]
df1['Volumenproduccion']= df1['Volumenproduccion'].str.replace(",","")
df2 = dgsiap_normb[4]
df2['Sembrada']= df2['Sembrada'].str.replace(",","")
df3 = dgsiap_normb[16]
df3['Rendimiento']= df3['Rendimiento'].str.replace(",","")

# Al quitar las comas ya podemos aplicar los cambios.
d1 = df1.astype(data_checker)
d2 = df2.astype(data_checker)

# Revisamos la situación del conjunto de datos faltante.
for g in df3['Rendimiento'].unique().tolist():
    try:
        float(g)
    except:
        print(g)

#¡NUM!


Vemos este caso especial. Ya que #¡NUM! es tipico de un excel cuando divides por cero. Ahora veremos que ocurrió en el conjunto de datos.

In [267]:
prob_df = df3[df3['Rendimiento']=="#¡NUM!"]
prob_df

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
28036,2009,26,Sonora,143,Moctezuma,2,Huásabas,32,Huásabas,2,...,Tonelada,6840000,Frijol,6.0,0.0,6.0,2.4,#¡NUM!,0.0,0.0
28044,2009,26,Sonora,143,Moctezuma,2,Huásabas,67,Villa Hidalgo,2,...,Tonelada,6840000,Frijol,38.0,0.0,38.0,15.2,#¡NUM!,0.0,0.0


Vemos, que en efecto, el error se debio por dividir por cero. Ya que el rendimiento, según el diccionario es

$$
    \dfrac{Volumenproduccion}{Cosechada}
$$
y vemos que es cero. Entonces lo que haremos es remplazarlos por valores nulos.

In [277]:
df3['Rendimiento'] = df3['Rendimiento'].replace("#¡NUM!", pd.NA)
df3.astype(data_checker)

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Precio,Valorproduccion
0,2009,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,873.0,873.0,0.0,23579.73,27.01,359.80,8483986.85
1,2009,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,9090000,Triticale forrajero en verde,341.0,341.0,0.0,12074.81,35.41,399.93,4829078.76
2,2009,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,15050000,Pastos y praderas,526.0,526.0,0.0,32612.00,62.00,343.20,11192438.40
3,2009,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,3.0,3.0,0.0,21.00,7.00,4650.00,97650.00
4,2009,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5940000,Cebolla,22.0,22.0,0.0,418.00,19.00,4000.00,1672000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35557,2009,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,7960000,Nuez,16.0,16.0,0.0,26.40,1.65,20000.00,528000.00
35558,2009,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,8170000,Pastos y praderas seco,50.0,50.0,0.0,300.00,6.00,1000.00,300000.00
35559,2009,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,9120000,Tuna,75.0,75.0,0.0,525.00,7.00,3200.00,1680000.00
35560,2009,32,Zacatecas,190,Jalpa,4,Jalpa,19,Jalpa,3,...,Tonelada,5050000,Agave,385.0,55.0,0.0,4400.00,80.00,500.00,2200000.00


Concluyendo aplicando esto para todos los conjuntos de datos.

In [283]:
parse_rules = {"Rendimiento": lambda x: x.replace("#¡NUM!", pd.NA),
                "Sembrada":lambda x: x.str.replace(",",""),
                "Volumenproduccion":lambda x: x.str.replace(",","")}

toparse_dfs = []
for k, df in enumerate(dgsiap_normb):
    dft = df.dtypes
    diff = dft[data_checker!=dft]
    if len(diff)>0:
        type_probs = diff.index.tolist()
        print("Conjunto de datos:",k)
        print("Columnas problematicas:", diff.index.tolist())
        for p in type_probs:
            df[p] = parse_rules[p](df[p])
        parse_df = df.astype(data_checker)
        toparse_dfs.append(parse_df)
    else:
        toparse_dfs.append(df)


Conjunto de datos: 3
Columnas problematicas: ['Volumenproduccion']
Conjunto de datos: 4
Columnas problematicas: ['Sembrada']
Conjunto de datos: 16
Columnas problematicas: ['Rendimiento']


Finalmente verificamos y preparamos el analisis de datos.

In [324]:
for k, df in enumerate(toparse_dfs):
    dft = df.dtypes
    diff = dft[data_checker!=dft]
    if len(diff)>0:
        type_probs = diff.index.tolist()
        print("Conjunto de datos:",k)
        print("Columnas problematicas:", diff.index.tolist())
        for p in type_probs:
            df[p] = parse_rules[p](df[p])
        parse_df = df.astype(data_checker)
        toparse_dfs.append(parse_df)
# Si no sale nada es que ya todos los tipos de datos coinciden. 


def norm2_um(df):
    crops = df['Nomcultivo'].unique().tolist()
    um = df['Nomunidad'].unique().tolist()
    um = set([u.lower() for u in um])
    par_detect = {v: re.search(r"\(.*\)", v.lower()).group(0) for v in crops if re.search(r"\(.*\)", v)}
    crop_w_um = []
    for crop in par_detect.keys():
        # Agregamos el caso particular
        if par_detect[crop][1:-1] in um or par_detect[crop][1:-1]=="m2":
            crop_w_um.append(crop)
    df['Nomcultivo'] = df['Nomcultivo'].apply(lambda x: re.sub(r"\(.*\)","", x) if x in set(crop_w_um) else x)
    return df


def normcol_branch(ref, branch, norm_col="Nomcultivo", norm_rule=norm2_um):
    catalog = ref.open(pars=[1, {"index_col":0}])
    ndf = []
    for file in branch:
        df = file.open(pars=[1, {"low_memory":False, "index_col":0}])
        cols = set(df.columns)
        if norm_col in cols:
            vdf = norm_rule(df)
            ndf.append(vdf)
        else:
            df.columns = catalog.index
            ndf.append(df)
    return ndf


parse_rules = {"Rendimiento": lambda x: x.replace("#¡NUM!", pd.NA),
                "Sembrada":lambda x: x.str.replace(",",""),
                "Volumenproduccion":lambda x: x.str.replace(",","")}


def vars_guide(data_vars, ref_df):
    df = pd.DataFrame(index = data_vars.keys(), columns = ['Tipo Pandas', 'Descripción'])
    for col in ref_df.columns:
        # print(col)
        var_type = catalog.loc[col, 'Tipo']
        df.loc[var_type] = [ref_df.dtypes[col], data_vars[var_type]]
    df=df.dropna(axis=0)
    return df

def parse_files(files, parse_rules, data_checker):
    toparse_dfs = []
    for k, df in enumerate(files):
        dft = df.dtypes
        diff = dft[data_checker!=dft]
        if len(diff)>0:
            type_probs = diff.index.tolist()
            print("Conjunto de datos:",k)
            print("Cnormcol_brancholumnas problematicas:", diff.index.tolist())
            for p in type_probs:
                df[p] = parse_rules[p](df[p])
            parse_df = df.astype(data_checker)
            toparse_dfs.append(parse_df)
        else:
            toparse_dfs.append(df)
    return toparse_dfs

from scripts.config import ROOT,DATA_DIR
from scripts.Files import FileTree, File

data_checker = pd.Series(index = df.columns, data=agro_dtypes)
data_checker
        

data_checker['Idmunicipio'] = agro_varsguide.loc['N', 'Tipo Pandas']
data_checker


def process_tree(tree, normcols="Nomcultivo", parse_rules=parse_rules):
    # Ajustamos columna Nomcultivo
    codebook = tree.root
    norm_branch = normcol_branch(codebook, tree.branch, norm_col=normcols, norm_rule=norm_um)
    # Creamos nuestra guia de vairables
    file_vars = tree.branch[0]
    ref_vars = file_vars.open(pars=[1, {"low_memory":False, "index_col":0}])
    vguide = vars_guide(tree.subroot, ref_vars)
    
    codes = codebook.open(pars=[1, {"index_col":0}])
    codes.drop(columns=["Long."], inplace=True)
    codes.columns = ['Tipo', 'Rangos válidos', 'Descripción', 'Observaciones']
    vars_name = "Variables_ProdAgro_mun.xlsx"
    code_name = "Codebook_ProdAgro_mun.xlsx"

    reference_path = ROOT / "references" # Al ser la versión final ira como una referencia.
    codepath = reference_path / code_name
    varspath = reference_path / vars_name
    vguide.to_excel(varspath)
    codes.to_excel(codepath)
    catlog = File(codepath, tree.source)
    guide = File(varspath, tree.source)

    agro_dtypes = []
    for col in df.columns:
        var_type = catalog.loc[col, 'Tipo']
        agro_dtypes.append(agro_varsguide.loc[var_type, 'Tipo Pandas'])

    data_checker = pd.Series(index = ref_vars.columns, data=agro_dtypes)
    data_checker['Idmunicipio'] = agro_varsguide.loc['N', 'Tipo Pandas']
    
    # Corregimos alguna otra columna pendiente.
    parse_branch = parse_files(norm_branch, parse_rules, data_checker)
    print(len(parse_branch))
    
    process_path = DATA_DIR / "process"
    bfile = []
    for n, pb in enumerate(parse_branch):
        file_name = f"ProdAgro_{n}_mun.csv"
        filepath = process_path / tree.source / file_name
        pb.to_csv(filepath, encoding="utf-8")
        bfile.append(File(filepath, tree.source))
    process_file = FileTree(catlog, bfile, source=tree.source)
    process_file.add_subroot(guide)
    return process_file


finaltest = process_tree(dgsiap)

Conjunto de datos: 3
Cnormcol_brancholumnas problematicas: ['Volumenproduccion']
Conjunto de datos: 4
Cnormcol_brancholumnas problematicas: ['Sembrada']
Conjunto de datos: 16
Cnormcol_brancholumnas problematicas: ['Rendimiento']
23


Finalmente hacemos las pruebas para verificar su uso.

In [326]:
froot = finaltest.root
fguide = finaltest.subroot
files = finaltest.branch

In [315]:
froot.open()

,Nombre\nde campo,Tipo,Rangos válidos,Descripción,Observaciones
0,Anio,N,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...
1,Idestado,N,"{1,…,32 }",Código que identifica a la entidad federativa ...,Clave geoestadística consecutiva que se asigna...
2,Nomestado,T,{CCC},Nombre oficial de la entidad federativa o estado,La entidad federativa es la unidad geográfica ...
3,Idddr,N,"{1,…,193}",Código que identifica al Distrito de Desarrol...,Clave interna que utiliza la Secretaría de Agr...
4,Nomddr,T,{CCC},Nombre que identifica al Distrito de Desarrol...,Nombre que utiliza la Secretaría de Agricultur...
5,Idcader,N,"{1,…,19}",Código que identifica al Centro de Apoyo al D...,Clave interna que utiliza la Secretaría de Agr...
6,Nomcader,T,{CCC},Nombre que identifica al Centro de Apoyo al D...,Nombre que utiliza la Secretaría de Agricultur...
7,Idmunicipio,T,"{1,….,570}",Código que identifica al municipio o delegació...,Unidad geográfica de la División territorial p...
8,Nommunicipio,T,{CCC},Nombre del municipio o delegación de producción,El municipio refiere a la unidad geográfica de...
9,Idciclo,N,"{1,….,3}",Código que identifica al ciclo agrícola,Código asignado en el catálogo de ciclo agrícola.


In [327]:
fguide.open()


,Unnamed: 0,Tipo Pandas,Descripción
0,T,str,Texto
1,N,int64,Numérico entero
2,D,float64,Numérico decimal


In [331]:
f = files[7]
f.open().dtypes


Unnamed: 0              int64
Anio                    int64
Idestado                int64
Nomestado                 str
Idddr                   int64
Nomddr                    str
Idcader                 int64
Nomcader                  str
Idmunicipio             int64
Nommunicipio              str
Idciclo                 int64
Nomcicloproductivo        str
Idmodalidad             int64
Nommodalidad              str
Idunidadmedida          int64
Nomunidad                 str
Idcultivo               int64
Nomcultivo                str
Sembrada              float64
Cosechada             float64
Siniestrada           float64
Volumenproduccion     float64
Rendimiento           float64
Precio                float64
Valorproduccion       float64
dtype: object

In [332]:
from scripts.config import ROOT
with open(ROOT / "preload" /"DGSIAP_PROCESS.pkl", "wb") as pdata:
    pickle.dump(finaltest, pdata)